### LÄSA IN DATA

In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

### BYGG SÄSONGSBASERAD DATASET (senaste säsong per spelare)

In [34]:
import os
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

# ==========================================
# 1. LÄS IN KAGGLE-DATAN
# ==========================================

players = pd.read_csv("players.csv")
apps = pd.read_csv("appearances.csv")
games = pd.read_csv("games.csv")

# ==========================================
# 2. LÄGG TILL SÄSONG PÅ VARJE APPEARANCE
# ==========================================

apps = apps.merge(
    games[["game_id", "season"]],
    on="game_id",
    how="left"
)

# ==========================================
# 3. SUMMERA STATS PER SPELARE + SÄSONG
# ==========================================

season_stats = (
    apps.groupby(["player_id", "season"])[
        ["goals", "assists", "minutes_played"]
    ]
    .sum()
    .reset_index()
)

# ==========================================
# 4. HITTA SENASTE SÄSONGEN FÖR VARJE SPELARE
# ==========================================

latest_season = (
    season_stats
    .sort_values("season")
    .groupby("player_id")
    .tail(1)
)

# Ta bara med spelare vars senaste säsong är efter 2020
latest_season = latest_season[latest_season["season"] > 2020]

# ==========================================
# 5. SLÅ IHOP MED PLAYER-INFORMATION
# ==========================================

df = players.merge(
    latest_season,
    on="player_id",
    how="inner"
)

# ==========================================
# 6. SKAPA AGE
# ==========================================

df["date_of_birth"] = pd.to_datetime(
    df["date_of_birth"],
    errors="coerce"
)

df["age"] = 2026 - df["date_of_birth"].dt.year

# ==========================================
# 7. VÄLJ ENDAST FEATURES + TARGET
# ==========================================

features = [
    "position",
    "age",
    "goals",
    "assists",
    "minutes_played",
    "current_club_domestic_competition_id"
]

target = "market_value_in_eur"

df_final = df[features + [target]].copy()

# Ta bort spelare utan target
df_final = df_final.dropna(subset=[target])

print(df_final.head())
print(df_final.shape)

# ==========================================
# 8. LADDA UPP TILL MONGODB (ny collection, rör inte gamla players_data)
# ==========================================

load_dotenv()

mongo_uri = os.getenv("MONGODB_URI")

client = MongoClient(mongo_uri)

db = client["football_data"]

new_collection = db["players_data_season"]

records = df_final.to_dict("records")

new_collection.delete_many({})
new_collection.insert_many(records)

print(
    f"{new_collection.count_documents({})} spelare sparade i MongoDB"
)

     position   age  goals  assists  minutes_played  \
0    Midfield  40.0      1        1             946   
1      Attack  45.0      1        0             144   
2  Goalkeeper  43.0      0        0              90   
3      Attack  43.0      2        1            1005   
4  Goalkeeper  44.0      0        0             720   

  current_club_domestic_competition_id  market_value_in_eur  
0                                  GB1             500000.0  
1                                  IT1            2000000.0  
2                                  DK1             100000.0  
3                                  PO1             300000.0  
4                                  NL1              75000.0  
(15308, 7)
15308 spelare sparade i MongoDB


In [ ]:
print(records[records['age'])

40.0
45.0
43.0
43.0
44.0
42.0
42.0
35.0
43.0
42.0


In [35]:
import os
import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient

# 1. Läs in miljövariabler från .env
load_dotenv()
mongo_uri = os.getenv("MONGODB_URI")

if not mongo_uri:
    raise ValueError("Kunde inte hitta MONGO_URI. Kontrollera att .env-filen finns och innehåller rätt variabel.")

# 2. Koppla upp mot MongoDB Atlas
client = MongoClient(mongo_uri)
db = client["football_data"]
collection = db["players_data_season"]

# 3. Hämta alla dokument och uteslut _id
data = list(collection.find({}, {"_id": 0}))

# 4. Gör om till DataFrame
df_from_db = pd.DataFrame(data)

# 5. Bygg X och y
target = 'market_value_in_eur'
y = df_from_db[target]
X_features = df_from_db.drop(columns=[target])

X = pd.get_dummies(X_features, drop_first=True)

print(df_from_db.shape)

(15308, 7)


### EDA

In [36]:

print(y.mean())

3432283.4465638883


# läsa in från mongo db

### TRÄNA / TESTA

In [37]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score

X_train_full, X_test, y_train_full, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

age_imputer = SimpleImputer(strategy='mean')       # eller 'median'
age_imputer.fit(X_train[['age']])                   # lär sig BARA från X_train

X_train['age'] = age_imputer.transform(X_train[['age']]) 
X_val['age']   = age_imputer.transform(X_val[['age']])
X_test['age']  = age_imputer.transform(X_test[['age']])

y_train_log = np.log1p(y_train)

params = {
    'max_depth': [None, 5, 10, 20, 30],
    'n_estimators': [50, 100, 150],
    'min_samples_split': [2, 5, 10]
}
clf = RandomForestRegressor(random_state=42)
gs = GridSearchCV(estimator=clf, param_grid=params, cv=2, n_jobs=-1, verbose=2)
gs.fit(X_train, y_train_log)
print("Bästa parametrar:", gs.best_params_) 
models = {
    "BaseLine" : DummyRegressor(strategy='mean'),
    'Linjär Regression' : LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    "Random Forest (Tuned)": gs.best_estimator_
}


for name, model in models.items():
    if name != "Random Forest (Tuned)":
        model.fit(X_train, y_train_log)

    # Prediktera och konvertera tillbaka till vanliga euro
    log_preds = model.predict(X_val)
    preds = np.expm1(log_preds)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)

    print(f'{name:20} | {rmse:12,.0f} € | {r2:6.3f}')


Fitting 2 folds for each of 45 candidates, totalling 90 fits
Bästa parametrar: {'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 150}
BaseLine             |    9,167,679 € | -0.077
Linjär Regression    |    6,307,473 € |  0.490
Random Forest        |    5,509,577 € |  0.611
Random Forest (Tuned) |    5,573,348 € |  0.602


In [38]:
test_pred_log = gs.best_estimator_.predict(X_test)
test_pred = np.expm1(test_pred_log)

final_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
final_r2 = r2_score(y_test, test_pred)

print('RMSE = ', final_rmse)
print('R2 = ', final_r2)


RMSE =  4791186.746060156
R2 =  0.6652891311494287


In [39]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 15308 entries, 0 to 15307
Data columns (total 39 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   age                                        15306 non-null  float64
 1   goals                                      15308 non-null  int64  
 2   assists                                    15308 non-null  int64  
 3   minutes_played                             15308 non-null  int64  
 4   position_Defender                          15308 non-null  bool   
 5   position_Goalkeeper                        15308 non-null  bool   
 6   position_Midfield                          15308 non-null  bool   
 7   position_Missing                           15308 non-null  bool   
 8   current_club_domestic_competition_id_ARG1  15308 non-null  bool   
 9   current_club_domestic_competition_id_AUS1  15308 non-null  bool   
 10  current_club_domestic_competition

In [40]:
import numpy as np
import pandas as pd

# 1. Skapa en tom rad med exakt samma kolumnnamn som träningsdatan
calle = pd.DataFrame(0, index=[0], columns=X_train.columns)

# 2. Fyll i siffervärden
calle['age'] = 25
calle['goals'] = 27
calle['assists'] = 8
calle['minutes_played'] = 3590  # t.ex. en hel säsong som ordinarie

# 3. Sätt en 1:a på hans position (Anfallare / Attack)
# Beroende på hur texten ser ut i er data heter den oftast 'position_Attack'
for col in calle.columns:
    if 'position' in col and 'Attack' in col:
        calle[col] = 1

# 4. Sätt en 1:a på ligan (Allsvenskan = SE1)
for col in calle.columns:
    if 'SE1' in col:
        calle[col] = 1

# 5. Låt modellen gissa (kom ihåg att omvandla från log-skala med expm1!)
basta_modell = gs.best_estimator_
calle_log_pred = basta_modell.predict(calle)
calle_varde = np.expm1(calle_log_pred)[0]

print(f"Spelare: Calle Pålsson")
print(f"Stats:   27 mål, 8 assist, Allsvenskan")
print(f"Uppskattat marknadsvärde: {calle_varde:,.0f} €")

Spelare: Calle Pålsson
Stats:   27 mål, 8 assist, Allsvenskan
Uppskattat marknadsvärde: 60,346,524 €
